# 🎓 Calculative Foundation: Linear Algebra in Student Performance Analysis
**Course / Project Duration:** 6 Hours | **Type:** Theory + Practical  
**Objective:** Analyzing student academic performance using vectors, matrix transformations, norms, projections, and eigenvalue/eigenvector decompositions.

## 1. Environment Setup & Data Preparation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.set_printoptions(precision=4, suppress=True)

# Dataset: 5 Students across 5 Core Subjects
students = ['Aarav', 'Bhavya', 'Chirag', 'Diya', 'Esha']
subjects = ['Math', 'Physics', 'Chemistry', 'CS', 'English']

data = np.array([
    [85, 90, 78, 92, 88],
    [70, 75, 80, 65, 72],
    [92, 88, 95, 90, 85],
    [60, 65, 70, 58, 62],
    [78, 82, 84, 80, 79]
])

df = pd.DataFrame(data, index=students, columns=subjects)
df.to_csv('student_scores.csv')
print("--- Student Performance Matrix (A) ---")
display(df)

## 📌 Part A: Vector Fundamentals
Each student $i$ is represented as a score vector $\vec{v}_i \in \mathbb{R}^5$.

In [2]:
v_aarav = data[0]
v_bhavya = data[1]

# 1. Norm-1 and Norm-2
norm1_aarav = np.linalg.norm(v_aarav, ord=1)
norm2_aarav = np.linalg.norm(v_aarav, ord=2)

norm1_bhavya = np.linalg.norm(v_bhavya, ord=1)
norm2_bhavya = np.linalg.norm(v_bhavya, ord=2)

# 2. Dot product and angle
dot_ab = np.dot(v_aarav, v_bhavya)
cos_theta = dot_ab / (norm2_aarav * norm2_bhavya)
cos_theta = np.clip(cos_theta, -1.0, 1.0)
angle_rad = np.arccos(cos_theta)
angle_deg = np.degrees(angle_rad)

# 3. Cross Product (3D - Math, Physics, Chemistry)
v_aarav_3d = v_aarav[:3]
v_bhavya_3d = v_bhavya[:3]
cross_ab = np.cross(v_aarav_3d, v_bhavya_3d)

# 4. Vector Projection of Aarav onto Bhavya
proj_scalar = dot_ab / (norm2_bhavya ** 2)
proj_a_on_b = proj_scalar * v_bhavya

print(f"Aarav Vector: {v_aarav}")
print(f"  L1 Norm (Sum of scores): {norm1_aarav}")
print(f"  L2 Norm (Euclidean length): {norm2_aarav:.4f}\n")

print(f"Bhavya Vector: {v_bhavya}")
print(f"  L1 Norm: {norm1_bhavya}")
print(f"  L2 Norm: {norm2_bhavya:.4f}\n")

print(f"Dot Product (Aarav . Bhavya): {dot_ab}")
print(f"Cosine Similarity: {cos_theta:.6f}")
print(f"Angle between vectors: {angle_deg:.4f}° ({angle_rad:.4f} rad)\n")

print(f"Cross Product in 3D (Math, Phys, Chem): {cross_ab}")
print(f"Projection of Aarav on Bhavya: {np.round(proj_a_on_b, 2)}")

### Interpretation of Part A
- **Norm-1 ($L_1$)**: Represents aggregate raw points. Aarav has 433 points vs Bhavya's 362 points.
- **Norm-2 ($L_2$)**: Euclidean magnitude, giving greater weight to peak subject scores.
- **Angle ($6.75^\circ$)**: The small angle between Aarav and Bhavya demonstrates extreme directional alignment ($\cos\theta \approx 0.993$). This indicates that their relative strengths across subjects follow an almost identical pattern, despite differences in overall magnitude.
- **Cross Product**: Produces an orthogonal normal vector $[1350, -1340, 75]$ representing the plane formed by Aarav and Bhavya's 3-subject profiles.
- **Projection**: Decomposes Aarav's performance along Bhavya's directional profile, indicating what Aarav's scores would look like scaled to Bhavya's exact ratio.

## 📌 Part B: Matrix Operations
Transforming and analyzing the full student score matrix $A$.

In [3]:
A = data.astype(float)

# 1. Matrix Addition (e.g., adding bonus points or comparing semester runs)
Bonus = np.full_like(A, 5.0)  # 5 bonus marks across all subjects
A_plus_Bonus = A + Bonus

# Matrix Multiplication: Subject correlation proxy A^T * A
ATA = np.dot(A.T, A)

# 2. Transpose
A_trans = A.T

# 3. Determinant
det_A = np.linalg.det(A)

# 4. Inverse
if abs(det_A) > 1e-6:
    inv_A = np.linalg.inv(A)
    is_invertible = True
else:
    inv_A = None
    is_invertible = False

print(f"Determinant of A: {det_A:.4f}")
print(f"Is Invertible? {is_invertible}\n")
print("Inverse of Matrix A (first 2 rows shown):")
print(inv_A[:2])

# Verify A * A_inv = I
identity_check = np.dot(A, inv_A)
print("\nVerification A * A^-1 (Identity check max absolute error):")
print(f"{np.max(np.abs(identity_check - np.eye(5))):.2e}")

### Interpretation of Part B
- **Determinant $\det(A) = 13125.0 \neq 0$**: The matrix is non-singular and full-rank ($	ext{rank}(A) = 5$). This confirms that no student's score profile is an exact linear combination of other students.
- **Invertibility**: Because $\det(A) \neq 0$, the unique inverse matrix $A^{-1}$ exists, allowing perfect reconstruction of inputs from any linear mapping.
- **Gram Matrix ($A^T A$)**: Measures feature-to-feature co-variance and interaction across all students.

## 📌 Part C: Linear Transformations & Geometry
Understanding dimension escalation: 2D line $\to$ 3D plane $\to$ $N$-D hyperplane.

In [4]:
# Visualizing 2D and 3D sub-spaces
fig = plt.figure(figsize=(14, 5))

# Subplot 1: 2D Line (Math vs Physics)
ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(df['Math'], df['Physics'], color='#1f77b4', s=80, zorder=5)
for i, txt in enumerate(students):
    ax1.annotate(txt, (df['Math'].iloc[i]+0.5, df['Physics'].iloc[i]+0.5), fontsize=10)

# Fit regression line (1D subspace in 2D)
m, c = np.polyfit(df['Math'], df['Physics'], 1)
x_vals = np.linspace(55, 95, 100)
ax1.plot(x_vals, m*x_vals + c, color='#d62728', linestyle='--', label=f'Line: Phys = {m:.2f}*Math + {c:.2f}')
ax1.set_title('2D Geometry: Line Subspace (Math vs Physics)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Math Score')
ax1.set_ylabel('Physics Score')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# Subplot 2: 3D Plane (Math, Physics, Chemistry)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(df['Math'], df['Physics'], df['Chemistry'], c='#2ca02c', s=80, depthshade=False)
for i, txt in enumerate(students):
    ax2.text(df['Math'].iloc[i], df['Physics'].iloc[i], df['Chemistry'].iloc[i], txt, fontsize=9)

# Fit 3D Plane: z = ax + by + d
X_plane = np.c_[df['Math'], df['Physics'], np.ones(5)]
coeff, _, _, _ = np.linalg.lstsq(X_plane, df['Chemistry'], rcond=None)
xx, yy = np.meshgrid(np.linspace(55, 95, 10), np.linspace(60, 95, 10))
zz = coeff[0]*xx + coeff[1]*yy + coeff[2]
ax2.plot_surface(xx, yy, zz, alpha=0.25, color='cyan')

ax2.set_title('3D Geometry: 2D Plane Subspace in 3D', fontsize=12, fontweight='bold')
ax2.set_xlabel('Math')
ax2.set_ylabel('Physics')
ax2.set_zlabel('Chemistry')

plt.tight_layout()
plt.savefig('geometry_subspaces.png', dpi=300)
plt.show()

### Geometric Foundations
- **Line (1D manifold in $\mathbb{R}^2$):** Defined by equation $w_1 x_1 + w_2 x_2 = b$. Divides 2D space into 2 half-spaces.
- **Plane (2D manifold in $\mathbb{R}^3$):** Defined by $w_1 x_1 + w_2 x_2 + w_3 x_3 = b$. Divides 3D space into 2 regions.
- **Hyperplane ($D-1$ manifold in $\mathbb{R}^D$):** For our 5-dimensional student space $\mathbb{R}^5$, a hyperplane is a 4-dimensional affine subspace given by:
  $$\vec{w}^T \vec{x} = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + w_5 x_5 = b$$
  In Data Science/ML (e.g., Support Vector Machines or Logistic Regression), hyperplanes act as linear decision boundaries separating categories (e.g., Pass vs Fail, Honors eligibility).

## 📌 Part D: Eigenvalues & Covariance Decomposition
Principal directions of variation and dimensionality reduction.

In [5]:
# Covariance matrix of subjects (columns)
cov_matrix = np.cov(A, rowvar=False)
cov_df = pd.DataFrame(cov_matrix, index=subjects, columns=subjects)

print("--- Covariance Matrix ---")
display(np.round(cov_df, 2))

# Compute Eigenvalues and Eigenvectors
evals, evecs = np.linalg.eigh(cov_matrix)

# Sort in descending order
sort_idx = np.argsort(evals)[::-1]
evals = evals[sort_idx]
evecs = evecs[:, sort_idx]

# Calculate explained variance
var_explained = evals / np.sum(evals) * 100
cum_var = np.cumsum(var_explained)

decomp_df = pd.DataFrame({
    'Eigenvalue (Variance)': evals,
    'Variance Explained (%)': var_explained,
    'Cumulative Variance (%)': cum_var
}, index=[f'PC{i+1}' for i in range(len(evals))])

print("\n--- Principal Components & Variance Explained ---")
display(np.round(decomp_df, 2))

# Visualizing Scree Plot
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar([f'PC{i+1}' for i in range(5)], var_explained, color='#4a69bd', alpha=0.85, label='Individual Variance')
ax.plot([f'PC{i+1}' for i in range(5)], cum_var, color='#e55039', marker='o', linewidth=2, label='Cumulative Variance')
ax.set_ylabel('Explained Variance Ratio (%)')
ax.set_title('Scree Plot: Principal Components Variance Explained', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
for i, v in enumerate(var_explained):
    ax.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
ax.legend(loc='center right')
plt.tight_layout()
plt.savefig('scree_plot.png', dpi=300)
plt.show()

### Interpretation of Part D
- **Dominant Eigenvalue ($PC_1 \approx 92.7\%$):** The first principal component captures **92.69%** of all variance in student performance. This reveals that student grades are overwhelmingly governed by a single general academic aptitude factor across STEM and humanities.
- **Second Component ($PC_2 \approx 6.58\%$):** Captures residual variance distinguishing STEM-heavy students from Humanities-heavy students.
- **Dimensionality Reduction:** The top 2 components capture **99.27%** of total data variation, meaning we can reduce the 5-subject space into a 2D plot with practically zero loss of analytical information.